In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hassan06/nslkdd")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Construct the correct path to the downloaded file
file_path = os.path.join(path, "KDDTrain+.txt")

df = pd.read_csv(file_path, header=None)
print(df.shape)
df.head()

In [ ]:
df = df.rename(columns={41: "label", 42: "difficulty"})
print(df.shape)
print(df["label"].value_counts().head(10))

df["is_attack"] = (df["label"] != "normal").astype(int)
print(df["is_attack"].value_counts())

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df.drop(columns=["label", "difficulty", "is_attack"])
X = pd.get_dummies(X, columns=[1, 2, 3])
X.columns = X.columns.astype(str)
X = X.astype(float)
y = df["is_attack"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_train[y_train == 0])  # learn "normal" only

pred = (model.predict(X_test) == -1).astype(int)  # -1 = weird
print(classification_report(y_test, pred, target_names=["normal", "attack"]))